# Evaluation and error analysis (#8)

Where the first DistilRoBERTa run fails, and why.

Run: macro-F1 **0.717** over the 34 trained categories, **0.595** if the 7 untrained
ones are counted as 0. Threshold 0.5, no tuning, no validation split.

The issue proposed a hypothesis worth testing directly: that misses come from clause
wording being split across two chunks. Section 3 tests it. It is wrong, and section 4
finds what does explain the failures.

## 0. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import textwrap

pd.set_option("display.width", 170)
pd.set_option("display.max_rows", 50)

metrics = pd.read_csv(ROOT / "reports/test_category_metrics.csv")
chunks = pd.read_parquet(ROOT / "data/processed/chunks.parquet")
labels = pd.read_parquet(ROOT / "data/processed/chunk_labels.parquet")
spans = pd.read_parquet(ROOT / "data/processed/label_audit/span_audit.parquet")
categories = pd.read_json(ROOT / "data/processed/contract_categories.jsonl", lines=True)

CATEGORY_COLUMNS = [c for c in labels.columns if c not in set(chunks.columns)]
print(f"{len(metrics)} evaluated categories | {len(chunks):,} chunks")

34 evaluated categories | 17,324 chunks


## 1. Per-category metrics

Evaluation is at contract level: did the model find this clause type anywhere in this
contract? Chunk probabilities are max-pooled per contract first, so a clause sitting in
a chunk overlap is not counted twice.

`support` is the number of test contracts that actually contain the category.

In [2]:
view = metrics.sort_values("f1", ascending=False)
print(view.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

                          category  precision  recall    f1  support  predicted_positive
                           Parties      1.000   0.980 0.990      102                 100
                     Document Name      1.000   0.951 0.975      102                  97
                         Insurance      0.941   1.000 0.970       32                  34
                     Governing Law      0.988   0.952 0.969       83                  80
                   Anti-Assignment      1.000   0.917 0.957       72                  66
                   Expiration Date      0.916   0.974 0.944       78                  83
                     License Grant      0.958   0.920 0.939       50                  48
                    Agreement Date      0.916   0.935 0.926       93                  95
                      Audit Rights      0.917   0.868 0.892       38                  36
                  Cap On Liability      0.854   0.932 0.891       44                  48
                     

## 2. False positives and false negatives

Derived from the metrics: `tp = recall x support`, `fp = predicted_positive - tp`,
`fn = support - tp`.

In [3]:
errors = metrics.copy()
errors["tp"] = (errors.recall * errors.support).round().astype(int)
errors["fp"] = errors.predicted_positive - errors.tp
errors["fn"] = errors.support - errors.tp

print(f"true positives : {errors.tp.sum()}")
print(f"false positives: {errors.fp.sum()}")
print(f"false negatives: {errors.fn.sum()}  "
      f"({100 * errors.fn.sum() / (errors.fn.sum() + errors.fp.sum()):.0f}% of all errors)")
print()
print("MOST MISSED (false negatives):")
print(errors.nlargest(6, "fn")[["category", "fn", "support", "recall"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("MOST FALSE ALARMS (false positives):")
print(errors.nlargest(6, "fp")[["category", "fp", "predicted_positive", "precision"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

true positives : 1016
false positives: 153
false negatives: 202  (57% of all errors)

MOST MISSED (false negatives):
                         category  fn  support  recall
                Change Of Control  16       26    0.38
               Volume Restriction  16       17    0.06
Competitive Restriction Exception  11       16    0.31
               Liquidated Damages  11       14    0.21
       Affiliate License-Licensee  10       12    0.17
        Post-Termination Services  10       29    0.66

MOST FALSE ALARMS (false positives):
                   category  fp  predicted_positive  precision
             Effective Date  19                  80       0.76
Termination For Convenience  18                  41       0.56
   Non-Transferable License  15                  32       0.53
         Uncapped Liability  11                  21       0.48
     Revenue/Profit Sharing   9                  40       0.78
             Agreement Date   8                  95       0.92


Misses outnumber false alarms 202 to 153. The model is **too cautious**, not too eager:
mean precision is 0.811 against mean recall 0.699, and 22 of 34 categories have
precision above recall.

That points at the 0.5 threshold rather than the model. Only 1.79% of chunk labels are
positive, so 0.5 is a high bar. Several categories have *perfect* precision with terrible
recall — `Joint Ip Ownership` is right on the 1 contract it flags and misses the other 6.

## 3. Testing the issue's hypothesis: are clauses split across chunks?

The span audit records, for every annotated clause, whether it fit inside one chunk
(`contained`), sat whole inside a chunk overlap (`in_overlap`), or was split with no
single chunk holding all of it (`crossing`).

If the hypothesis holds, categories with more `crossing` spans should score worse.

In [4]:
test_spans = spans[spans.split == "test"]
kinds = test_spans.groupby("category")["kind"].value_counts().unstack(fill_value=0)
kinds["n_spans"] = kinds.sum(axis=1)
kinds["pct_crossing"] = (100 * kinds.get("crossing", 0) / kinds.n_spans).round(1)

split_check = errors.merge(kinds.reset_index()[["category", "n_spans", "pct_crossing"]],
                           on="category", how="left")

print(f"test spans split across chunks: {100 * (test_spans.kind == 'crossing').mean():.1f}%")
print()
print(f"correlation, % spans split vs F1:     {split_check.pct_crossing.corr(split_check.f1):+.3f}")
print(f"correlation, % spans split vs recall: {split_check.pct_crossing.corr(split_check.recall):+.3f}")
print()
print("WORST 8 CATEGORIES - how many of their spans are actually split?")
print(split_check.nsmallest(8, "f1")[["category", "f1", "recall", "pct_crossing", "n_spans"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

test spans split across chunks: 2.0%



correlation, % spans split vs F1:     +0.004
correlation, % spans split vs recall: +0.018

WORST 8 CATEGORIES - how many of their spans are actually split?
                         category   f1  recall  pct_crossing  n_spans
               Volume Restriction 0.09    0.06          2.90       35
          No-Solicit Of Customers 0.20    0.14          0.00       15
               Joint Ip Ownership 0.25    0.14          0.00       14
       Affiliate License-Licensee 0.29    0.17          3.70       27
               Liquidated Damages 0.33    0.21          4.30       23
Competitive Restriction Exception 0.48    0.31          0.00       27
                Change Of Control 0.51    0.38          3.20       62
               Uncapped Liability 0.59    0.77         12.50       16


**The hypothesis does not hold.** Correlation is +0.004 — no relationship at all.

Only 2% of test spans are split in the first place, and the worst categories are not the
split ones: `No-Solicit Of Customers`, `Joint Ip Ownership` and
`Competitive Restriction Exception` have **0%** of their spans split and still score
0.20-0.48.

This is a useful negative result. It says **do not spend effort revisiting chunking** —
the 510/128 strategy from #22 is not what is holding the model back.

## 4. What does explain the failures?

Testing training signal instead.

In [5]:
train_rows = labels[labels.split == "train"]
signal = pd.DataFrame({
    "category": CATEGORY_COLUMNS,
    "train_contracts": train_rows.groupby("contract_id")[CATEGORY_COLUMNS].max()
                                 .fillna(False).sum().values,
    "train_chunks": train_rows[CATEGORY_COLUMNS].fillna(False).sum().values,
})
analysis = errors.merge(signal, on="category")

print("CORRELATION WITH F1")
for column, label in [("train_chunks", "positive train chunks"),
                      ("train_contracts", "positive train contracts"),
                      ("support", "positive test contracts")]:
    print(f"  {label:<28} {analysis[column].corr(analysis.f1):+.3f}")
print(f"  {'% spans split across chunks':<28} "
      f"{split_check.pct_crossing.corr(split_check.f1):+.3f}   <- not the problem")

CORRELATION WITH F1
  positive train chunks        +0.715
  positive train contracts     +0.669
  positive test contracts      +0.646
  % spans split across chunks  +0.004   <- not the problem


In [6]:
print("WORST 8 - training signal received:")
print(analysis.nsmallest(8, "f1")[["category", "f1", "recall", "train_contracts", "train_chunks"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))
print()
print("BEST 5 for contrast:")
print(analysis.nlargest(5, "f1")[["category", "f1", "recall", "train_contracts", "train_chunks"]]
      .to_string(index=False, float_format=lambda v: f"{v:.2f}"))

WORST 8 - training signal received:
                         category   f1  recall  train_contracts  train_chunks
               Volume Restriction 0.09    0.06               65           155
          No-Solicit Of Customers 0.20    0.14               27            54
               Joint Ip Ownership 0.25    0.14               39            98
       Affiliate License-Licensee 0.29    0.17               47           123
               Liquidated Damages 0.33    0.21               47           110
Competitive Restriction Exception 0.48    0.31               60           122
                Change Of Control 0.51    0.38               95           217
               Uncapped Liability 0.59    0.77               98           194

BEST 5 for contrast:
       category   f1  recall  train_contracts  train_chunks
        Parties 0.99    0.98              407           548
  Document Name 0.97    0.95              408           445
      Insurance 0.97    1.00              134           297


**Training volume is the driver**: +0.715 against positive training chunks, compared with
+0.004 for chunk splitting.

The best categories had 300-600 positive training chunks. The worst had 50-160.

## 5. The outlier: Volume Restriction

`Volume Restriction` breaks the pattern. It has 65 training contracts and 155 positive
chunks — more than `Joint Ip Ownership` or `No-Solicit Of Customers` — yet scores the
worst F1 of all 34 at 0.091, finding 1 of 17.

Comparing its clause text against `Insurance`, which has similar span lengths but scores
0.97, shows why.

In [7]:
def show_examples(category, n=3, width=150):
    rows = categories[(categories.category == category)
                      & (categories.split == "train") & categories.present]
    texts = [answer["text"] for row in rows.answers for answer in row]
    lengths = pd.Series([len(t.split()) for t in texts])
    print(f"=== {category} ===  {len(rows)} contracts, {len(texts)} spans, "
          f"median {int(lengths.median())} words")
    for text in texts[:n]:
        print("   >", textwrap.shorten(" ".join(text.split()), width))
    print()


show_examples("Insurance")
show_examples("Volume Restriction")

=== Insurance ===  134 contracts, 443 spans, median 38 words
   > Company will carry a reasonable amount of product liability insurance through a reasonably acceptable products liability insurance company and [...]
   > as an insurance company in its respective jurisdiction of incorporation and is duly licensed or authorized as an insurer in each other [...]
   > Each Insurance Subsidiary is in compliance with and conducts its businesses in conformity with all applicable insurance laws and regulations of [...]

=== Volume Restriction ===  65 contracts, 136 spans, median 38 words
   > During the term of this Agreement, CONSULTANT shall make himself available on not more than [*****] days for television and radio commercials, [...]
   > The Company will compensate Consultant at the rate of $400 per hour (19 hours cap monthly; anything over these hrs must be preapproved by [...]
   > The foregoing obligations of Wildform, following the initial 30 day period, shall be limited to a maximum 

`Insurance` clauses all contain the word **insurance**. The model has a reliable lexical
anchor.

`Volume Restriction` clauses share a *concept* — a cap on quantity — with no shared
vocabulary at all: "not more than [*****] days", "19 hours cap monthly", "a maximum of
twenty (20) hours". Nothing lexical ties them together, and the actual number is often
redacted to `[*****]`, removing even the quantity.

So this is not a data-volume problem. It is a genuinely harder category, defined by
meaning rather than wording. More training data would help less here than elsewhere.

## Conclusions

**1. The threshold is the biggest single lever, not the model.**
Misses outnumber false alarms 202 to 153, and 22 of 34 categories are more precise than
they are sensitive. Several have perfect precision with recall under 0.2 — the model
recognises those clauses and refuses to commit at 0.5. This is #7, and it needs held-out
data to tune honestly.

**2. Chunking is not the problem.** Correlation +0.004, and only 2% of spans are split.
The 510/128 strategy from #22 should be left alone.

**3. Training volume explains most of the variation** (+0.715). The weak categories are
the rare ones. Options are resampling, class weights (`pos_weight` is already plumbed
through and unused), or accepting a documented floor on rare categories.

**4. `Volume Restriction` is genuinely hard**, not under-trained. It is defined
semantically with no lexical anchor, and redaction removes the quantities. Worth calling
out as a known limitation rather than treating as a bug.

**5. The 7 untrained categories** have no positives in train and score 0. Report
macro-F1 over 41 as well as 34, so this is not hidden.

### Not covered here

Per-contract inspection of individual false positives and negatives needs
`test_contract_predictions.parquet` and `test_chunk_predictions.parquet` from the Colab
run, which are not in the repo (gitignored, ~300 KB). Download them from Drive at
`cuad-runs/distilroberta-cuad/` to extend this with named examples of each error type.